# GAIR/LIMA Gemini Translation Pipeline

Downloads GAIR/lima with Hugging Face authentication, translates every conversation with three rotating Gemini API keys, enforces per-key quotas, continues after failures, and writes JSON outputs.

Required environment variables: `GEMINI_API_KEY_1`, `GEMINI_API_KEY_2`, `GEMINI_API_KEY_3`, and `HF_TOKEN`. A local `.env` file is supported.

In [1]:
%pip install -q google-genai huggingface_hub python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
from huggingface_hub import hf_hub_download

load_dotenv()
ROOT = Path.cwd()
PROMPT_FILE = ROOT / 'translation_prompt.md'
SOURCE_JSON = ROOT / 'lima_records.json'
TRANSLATIONS_JSON = ROOT / 'lima_translations.json'
QUOTA_JSON = ROOT / 'gemini_quota_state.json'
TARGET_LANGUAGE = os.getenv('TARGET_LANGUAGE', 'Nepali')
MODEL = os.getenv('GEMINI_MODEL', 'gemini-3.5-flash-lite')
MAX_RECORDS = None  # Use an integer for a small test run.
REFRESH_SOURCE = False  # Set True only when intentionally refreshing the saved dataset snapshot.
# Keep this ordered list as the reproducibility record for this run's key pool.
KEY_NAMES = ('Gemini_api', 'Gemini_api_RL', 'Gemini_api_CH', 'Gemini_api_YN')
GEMINI_API_KEYS = [(name, os.getenv(name)) for name in KEY_NAMES]
HF_token = os.getenv('HF_token') or os.getenv('HF_token')
if any(not value for _, value in GEMINI_API_KEYS):
    raise RuntimeError(f"Set all Gemini API keys configured in KEY_NAMES: {', '.join(KEY_NAMES)}.")
if not HF_token:
    raise RuntimeError('Set HF_token before downloading GAIR/lima.')
TEMPERATURE, TOP_P, TOP_K, MAX_OUTPUT_TOKENS = 0.2, 0.95, 40, 4096
REQUESTS_PER_MINUTE, REQUESTS_PER_DAY = 14, 499
print(f'Using {len(GEMINI_API_KEYS)} keys, model={MODEL}, target={TARGET_LANGUAGE}')

Using 4 keys, model=gemini-3.5-flash-lite, target=Nepali


In [3]:
prompt_markdown = PROMPT_FILE.read_text(encoding='utf-8')
prompt_start = prompt_markdown.index('## Prompt:')
fence_start = prompt_markdown.index('```', prompt_start) + 3
fence_end = prompt_markdown.index('```', fence_start)
PROMPT_TEMPLATE = prompt_markdown[fence_start:fence_end].strip('\r\n')
if '{target_language}' not in PROMPT_TEMPLATE or '{source}' not in PROMPT_TEMPLATE:
    raise ValueError('The unchanged prompt must contain both placeholders.')

if SOURCE_JSON.exists() and not REFRESH_SOURCE:
    records = json.loads(SOURCE_JSON.read_text(encoding='utf-8'))
    print(f'Loaded {len(records)} saved source records from {SOURCE_JSON.name}')
else:
    dataset_path = hf_hub_download(repo_id='GAIR/lima', filename='train.jsonl', repo_type='dataset', token=HF_token)
    with open(dataset_path, encoding='utf-8') as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    SOURCE_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved {len(records)} source records to {SOURCE_JSON.name}')
if MAX_RECORDS is not None:
    records = records[:MAX_RECORDS]
print(f'Using {len(records)} source records for this run')

Loaded 1030 saved source records from lima_records.json
Using 1030 source records for this run


In [4]:
print(PROMPT_TEMPLATE)

Translate the following instruction-following conversation into {target_language}.

Rules:
- Translate BOTH the HUMAN and ASSISTANT turns completely. Do not leave any turn in the source language.
- Keep the speaker labels (HUMAN:, ASSISTANT:) exactly as-is, untranslated.
- Write ONLY in {target_language}. Do not add English glosses, translations, or parenthetical originals next to translated words.
- Proper nouns, brand/product names, and specialized technical terms with no standard {target_language} equivalent may remain in English. Ordinary vocabulary must be translated.
- Preserve all markdown formatting, line breaks, and structure exactly.
- Inside code blocks, translate only user-facing strings and natural-language comments; preserve identifiers, syntax, file paths, and URLs.
- Preserve numbers without changing their values.
- Translate naturally and idiomatically for {target_language}; preserve the original tone.
- Do not answer the conversation, add commentary, or wrap output in

In [5]:
def conversation_to_text(record):
    turns = []
    for index, turn in enumerate(record.get('conversations', [])):
        if isinstance(turn, dict):
            speaker = str(turn.get('from', 'unknown')).upper()
            content = str(turn.get('value', turn.get('content', '')))
        else:
            speaker = 'HUMAN' if index % 2 == 0 else 'ASSISTANT'
            content = str(turn)
        turns.append(f'{speaker}: {content}')
    return '\n\n'.join(turns)

def render_prompt(source):
    return PROMPT_TEMPLATE.replace('{target_language}', TARGET_LANGUAGE).replace('{source}', source)

def clean_text(text):
    match = re.fullmatch(r'\s*```(?:text|markdown)?\s*\n?(.*?)\n?```\s*', text, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else text.strip()

def load_state():
    today = datetime.now(timezone.utc).date().isoformat()
    if QUOTA_JSON.exists():
        state = json.loads(QUOTA_JSON.read_text(encoding='utf-8'))
        if state.get('date') == today:
            return state
    return {'date': today, 'keys': {name: {'count': 0, 'timestamps': []} for name, _ in GEMINI_API_KEYS}}

def save_state():
    QUOTA_JSON.write_text(json.dumps(quota_state, indent=2), encoding='utf-8')

def reserve_slot(key_name):
    while True:
        key = quota_state['keys'][key_name]
        now = time.time()
        key['timestamps'] = [stamp for stamp in key['timestamps'] if now - stamp < 60]
        if key['count'] >= REQUESTS_PER_DAY:
            return False
        if len(key['timestamps']) < REQUESTS_PER_MINUTE:
            key['timestamps'].append(now)
            key['count'] += 1
            save_state()
            return True
        delay = max(0.1, 60 - (now - min(key['timestamps'])))
        print(f'{key_name}: waiting {delay:.1f}s for minute quota')
        time.sleep(delay)

quota_state = load_state()
for key_name, _ in GEMINI_API_KEYS:
    quota_state['keys'].setdefault(key_name, {'count': 0, 'timestamps': []})
save_state()

In [6]:
from concurrent.futures import ThreadPoolExecutor
from queue import Empty, Queue
from threading import Lock


def translate(client, source):
    chat = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        ),
    )
    response = chat.send_message(render_prompt(source))
    if not response.text:
        raise RuntimeError('The model returned no text.')
    usage = getattr(response, 'usage_metadata', None)
    return clean_text(response.text), {
        'prompt_tokens': getattr(usage, 'prompt_token_count', None),
        'output_tokens': getattr(usage, 'candidates_token_count', None),
        'total_tokens': getattr(usage, 'total_token_count', None),
    }


results = json.loads(TRANSLATIONS_JSON.read_text(encoding='utf-8')) if TRANSLATIONS_JSON.exists() else []
successful = {item['index'] for item in results if item.get('status') == 'success'}
clients = {name: genai.Client(api_key=value) for name, value in GEMINI_API_KEYS}
work_queue = Queue()
for index, record in enumerate(records):
    if index not in successful:
        work_queue.put(index)

quota_lock = Lock()
results_lock = Lock()
disabled_keys = set()


def reserve_concurrent_slot(key_name):
    while True:
        with quota_lock:
            if key_name in disabled_keys:
                return False
            key = quota_state['keys'][key_name]
            now = time.time()
            key['timestamps'] = [stamp for stamp in key['timestamps'] if now - stamp < 60]
            if key['count'] >= REQUESTS_PER_DAY:
                return False
            if len(key['timestamps']) < REQUESTS_PER_MINUTE:
                key['timestamps'].append(now)
                key['count'] += 1
                save_state()
                return True
            delay = max(0.1, 60 - (now - min(key['timestamps'])))
        print(f'{key_name}: waiting {delay:.1f}s for minute quota')
        time.sleep(delay)


def save_result(item):
    with results_lock:
        updated = [old for old in results if old.get('index') != item['index']]
        updated.append(item)
        updated.sort(key=lambda old: old['index'])
        results[:] = updated
        TRANSLATIONS_JSON.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')


def worker(key_name):
    client = clients[key_name]
    while True:
        try:
            index = work_queue.get_nowait()
        except Empty:
            return
        try:
            if not reserve_concurrent_slot(key_name):
                work_queue.put(index)
                return
            source = conversation_to_text(records[index])
            item = {
                'index': index,
                'source': records[index],
                'source_text': source,
                'key': key_name,
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            }
            started = time.perf_counter()
            item['translation'], item['usage'] = translate(client, source)
            item.update(status='success', elapsed_seconds=round(time.perf_counter() - started, 3))
            save_result(item)
            print(f'[{index + 1}/{len(records)}] translated concurrently with {key_name}')
        except Exception as error:
            # An empty response is record-specific and retryable; keep the key working on the queue.
            disable_key = not (isinstance(error, RuntimeError) and str(error) == 'The model returned no text.')
            if disable_key:
                with quota_lock:
                    disabled_keys.add(key_name)
            item = {
                'index': index,
                'source': records[index],
                'source_text': conversation_to_text(records[index]),
                'key': key_name,
                'status': 'failed',
                'error': f'{type(error).__name__}: {error}',
                'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            }
            save_result(item)
            status = 'was disabled' if disable_key else 'remains active for other records'
            print(f'[{index + 1}/{len(records)}] {key_name} failed and {status}: {error}')
        finally:
            work_queue.task_done()


with ThreadPoolExecutor(max_workers=len(GEMINI_API_KEYS)) as executor:
    list(executor.map(worker, (name for name, _ in GEMINI_API_KEYS)))

save_state()
print(f'Saved {len(results)} records to {TRANSLATIONS_JSON.name}; disabled keys: {sorted(disabled_keys) or "none"}')

[605/1030] translated concurrently with Gemini_api_RL
[931/1030] translated concurrently with Gemini_api_RL
[701/1030] translated concurrently with Gemini_api_CH
[932/1030] translated concurrently with Gemini_api_RL
[933/1030] translated concurrently with Gemini_api_CH
[934/1030] translated concurrently with Gemini_api_RL
[702/1030] translated concurrently with Gemini_api_YN
[936/1030] translated concurrently with Gemini_api_RL
[937/1030] translated concurrently with Gemini_api_YN
[939/1030] translated concurrently with Gemini_api_YN
[940/1030] translated concurrently with Gemini_api_YN
[935/1030] translated concurrently with Gemini_api_CH
[941/1030] translated concurrently with Gemini_api_YN
[938/1030] translated concurrently with Gemini_api_RL
[942/1030] translated concurrently with Gemini_api_CH
[944/1030] translated concurrently with Gemini_api_RL
[943/1030] translated concurrently with Gemini_api_YN
[945/1030] translated concurrently with Gemini_api_CH
[947/1030] translated concur

In [7]:
print(json.dumps({
    'total': len(results),
    'successful': sum(item.get('status') == 'success' for item in results),
    'failed': sum(item.get('status') == 'failed' for item in results),
    'source_json': str(SOURCE_JSON),
    'translations_json': str(TRANSLATIONS_JSON),
    'quota_json': str(QUOTA_JSON),
}, indent=2))

{
  "total": 1028,
  "successful": 1025,
  "failed": 3,
  "source_json": "C:\\Projects\\Dataset Translator\\lima_records.json",
  "translations_json": "C:\\Projects\\Dataset Translator\\lima_translations.json",
  "quota_json": "C:\\Projects\\Dataset Translator\\gemini_quota_state.json"
}
